# Distributed Application using MapReduce
## Practical: Process a System Log File using Hadoop MapReduce on Google Colab

---
### What this practical does:
- Installs Java and Hadoop on Google Colab
- Creates a sample system log file
- Writes Mapper and Reducer in Python (Hadoop Streaming)
- Runs a MapReduce job to **count occurrences of each log level** (INFO, ERROR, WARN, DEBUG)
- Displays the final output

### Log file format used:
```
2024-01-01 10:00:01 INFO  UserService - User logged in
2024-01-01 10:00:02 ERROR DBService  - Connection failed
2024-01-01 10:00:03 WARN  AuthService - Token expiring
```
**Goal:** Count how many times each log level (INFO, ERROR, WARN, DEBUG) appears in the log file.


## STEP 1 - Install Java (required for Hadoop)

In [ ]:
# Install Java 8 (Hadoop requires Java)
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
!java -version

## STEP 2 - Download and Install Hadoop

In [ ]:
# Download Hadoop 3.3.6
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!ls hadoop-3.3.6/

In [ ]:
# Set Hadoop environment variables
import os
os.environ['HADOOP_HOME'] = '/content/hadoop-3.3.6'
os.environ['JAVA_HOME']   = '/usr/lib/jvm/java-8-openjdk-amd64'
os.environ['PATH']        = os.environ['PATH'] + ':/content/hadoop-3.3.6/bin:/content/hadoop-3.3.6/sbin'

# Verify Hadoop installation
!hadoop version

## STEP 3 - Create a Sample System Log File

In [ ]:
# Create a sample system log file (system_log.txt)
log_data = """2024-01-01 10:00:01 INFO  UserService - User admin logged in successfully
2024-01-01 10:00:02 ERROR DBService - Database connection failed: timeout
2024-01-01 10:00:03 WARN  AuthService - Authentication token is expiring soon
2024-01-01 10:00:04 INFO  FileService - File upload completed successfully
2024-01-01 10:00:05 DEBUG Scheduler - Cron job triggered at 10:00:05
2024-01-01 10:00:06 ERROR NetworkService - Unable to reach host 192.168.1.1
2024-01-01 10:00:07 INFO  UserService - User john logged out
2024-01-01 10:00:08 WARN  DiskService - Disk usage above 80 percent
2024-01-01 10:00:09 INFO  APIService - GET /api/users returned 200 OK
2024-01-01 10:00:10 ERROR DBService - Query execution failed: syntax error
2024-01-01 10:00:11 DEBUG CacheService - Cache miss for key user_123
2024-01-01 10:00:12 INFO  AuthService - New session created for user alice
2024-01-01 10:00:13 WARN  MemoryService - Memory usage above 75 percent
2024-01-01 10:00:14 INFO  FileService - File download started by user bob
2024-01-01 10:00:15 ERROR AuthService - Login failed for user unknown_user
2024-01-01 10:00:16 DEBUG Scheduler - Background task completed in 120ms
2024-01-01 10:00:17 INFO  APIService - POST /api/login returned 200 OK
2024-01-01 10:00:18 WARN  NetworkService - High latency detected: 500ms
2024-01-01 10:00:19 INFO  DBService - Database backup completed successfully
2024-01-01 10:00:20 ERROR FileService - File not found: report_2024.pdf
2024-01-01 10:00:21 INFO  UserService - Password changed for user charlie
2024-01-01 10:00:22 DEBUG CacheService - Cache hit for key product_456
2024-01-01 10:00:23 WARN  DiskService - Disk cleanup recommended
2024-01-01 10:00:24 INFO  APIService - DELETE /api/session returned 200 OK
2024-01-01 10:00:25 ERROR NetworkService - SSL certificate expired
"""

with open('system_log.txt', 'w') as f:
    f.write(log_data)

print("Log file created: system_log.txt")
print("\nFirst 5 lines of the log file:")
!head -5 system_log.txt

## STEP 4 - Write the Mapper (Python)

The Mapper reads each log line, extracts the log level (INFO/ERROR/WARN/DEBUG), and emits `(log_level, 1)`.

In [ ]:
# Write mapper.py
mapper_code = '''#!/usr/bin/env python3
import sys

# Mapper: reads each log line, extracts log level, emits (log_level, 1)
for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    parts = line.split()
    # Log format: DATE TIME LEVEL SERVICE - MESSAGE
    # Index:       0    1    2     3       4  5...
    if len(parts) >= 3:
        log_level = parts[2].strip()   # Extract log level (INFO/ERROR/WARN/DEBUG)
        print(f"{log_level}\\t1")       # Emit (log_level, 1)
'''

with open('mapper.py', 'w') as f:
    f.write(mapper_code)

!chmod +x mapper.py
print("mapper.py created successfully")
print("\n--- mapper.py content ---")
!cat mapper.py

## STEP 5 - Write the Reducer (Python)

The Reducer receives sorted `(log_level, 1)` pairs, sums up the counts for each log level, and emits `(log_level, total_count)`.

In [ ]:
# Write reducer.py
reducer_code = '''#!/usr/bin/env python3
import sys

# Reducer: sums up counts for each log level
current_key = None
current_count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    key, value = line.split("\\t", 1)
    value = int(value)

    if current_key == key:
        current_count += value          # Same key: accumulate count
    else:
        if current_key is not None:
            print(f"{current_key}\\t{current_count}")  # Emit previous key result
        current_key = key
        current_count = value

# Emit the last key
if current_key is not None:
    print(f"{current_key}\\t{current_count}")
'''

with open('reducer.py', 'w') as f:
    f.write(reducer_code)

!chmod +x reducer.py
print("reducer.py created successfully")
print("\n--- reducer.py content ---")
!cat reducer.py

## STEP 6 - Test Mapper and Reducer Locally (without Hadoop)

Before running on Hadoop, test the pipeline using Unix pipes to simulate MapReduce.

In [ ]:
# Test: Mapper output (intermediate key-value pairs)
print("=== MAPPER OUTPUT (intermediate key-value pairs) ===")
!cat system_log.txt | python3 mapper.py

In [ ]:
# Test: Full MapReduce pipeline locally (Map -> Sort -> Reduce)
print("=== FULL MapReduce PIPELINE (local simulation) ===")
print("Input -> Mapper -> Sort (shuffle) -> Reducer -> Output")
print()
!cat system_log.txt | python3 mapper.py | sort | python3 reducer.py

## STEP 7 - Upload Log File to HDFS and Run MapReduce Job

In [ ]:
# Configure Hadoop for standalone (local) mode
import subprocess

# Set JAVA_HOME inside Hadoop config
hadoop_env_path = '/content/hadoop-3.3.6/etc/hadoop/hadoop-env.sh'
with open(hadoop_env_path, 'a') as f:
    f.write('\nexport JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64\n')

print("Hadoop configured for standalone mode")
!hadoop version

In [ ]:
# Create HDFS input directory and upload log file
!hadoop fs -mkdir -p /loginput
!hadoop fs -put system_log.txt /loginput/
!hadoop fs -ls /loginput
print("\nLog file uploaded to HDFS successfully")

In [ ]:
# Remove output directory if it exists (for re-runs)
!hadoop fs -rm -r /logoutput 2>/dev/null || true

# Run the MapReduce job using Hadoop Streaming
print("=== RUNNING MapReduce JOB ON HADOOP ===")
!hadoop jar /content/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-*.jar \
    -input /loginput/system_log.txt \
    -output /logoutput \
    -mapper mapper.py \
    -reducer reducer.py \
    -file mapper.py \
    -file reducer.py

## STEP 8 - View the Output

In [ ]:
# View the MapReduce output from HDFS
print("=== MapReduce OUTPUT (Log Level Counts) ===")
!hadoop fs -cat /logoutput/part-00000

In [ ]:
# List output files in HDFS
print("=== Output files in HDFS ===")
!hadoop fs -ls /logoutput

## STEP 9 - Visualize the Results

In [ ]:
import matplotlib.pyplot as plt
import subprocess

# Read output from HDFS
result = subprocess.run(
    ['hadoop', 'fs', '-cat', '/logoutput/part-00000'],
    capture_output=True, text=True
)

# Parse the output
log_counts = {}
for line in result.stdout.strip().split('\n'):
    if line.strip():
        parts = line.split('\t')
        if len(parts) == 2:
            log_counts[parts[0]] = int(parts[1])

print("Log Level Counts:")
for level, count in sorted(log_counts.items()):
    print(f"  {level:10s} : {count}")

# Bar chart
colors = {'INFO': 'steelblue', 'ERROR': 'crimson', 'WARN': 'orange', 'DEBUG': 'green'}
bar_colors = [colors.get(k, 'gray') for k in log_counts.keys()]

plt.figure(figsize=(8, 5))
plt.bar(log_counts.keys(), log_counts.values(), color=bar_colors, edgecolor='black')
plt.title('System Log Level Counts (MapReduce Output)', fontsize=14, fontweight='bold')
plt.xlabel('Log Level', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(fontsize=11)
for i, (k, v) in enumerate(log_counts.items()):
    plt.text(i, v + 0.1, str(v), ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('log_level_counts.png', dpi=150)
plt.show()
print("Chart saved as log_level_counts.png")

---
# IMPORTANT NOTES FOR ORAL EXAM
## DISTRIBUTED APPLICATION USING MapReduce - LOG FILE PROCESSING

---

## WHAT IS THIS PRACTICAL ABOUT?
This practical counts the number of occurrences of each **log level** (INFO, ERROR, WARN, DEBUG)  
from a system log file using **Hadoop MapReduce** — a distributed processing framework.

Files in this practical:
- `mapper.py`       — The Mapper (extracts log level, emits `(log_level, 1)`)
- `reducer.py`      — The Reducer (sums counts per log level)
- `system_log.txt`  — Input log file uploaded to HDFS

---

## WHAT IS MapReduce?
MapReduce is a programming model for processing large datasets in parallel across a cluster.

| Phase | What happens |
|---|---|
| **Map** | Each line is processed independently. Emits `(key, value)` pairs. |
| **Shuffle & Sort** | Framework groups all values for the same key together. |
| **Reduce** | Aggregates values for each key. Emits final `(key, result)`. |

---

## FULL MapReduce FLOW FOR THIS PRACTICAL

**INPUT (system_log.txt):**
```
2024-01-01 10:00:01 INFO  UserService - User logged in
2024-01-01 10:00:02 ERROR DBService - Connection failed
2024-01-01 10:00:03 WARN  AuthService - Token expiring
2024-01-01 10:00:04 INFO  FileService - Upload complete
```

**AFTER MAP PHASE (intermediate output):**
```
(INFO,  1)
(ERROR, 1)
(WARN,  1)
(INFO,  1)
```

**AFTER SHUFFLE & SORT (grouped by key):**
```
(ERROR, [1])
(INFO,  [1, 1])
(WARN,  [1])
```

**AFTER REDUCE PHASE (final output):**
```
DEBUG    3
ERROR    5
INFO     9
WARN     4
```

---

## CODE EXPLANATION

### Mapper (mapper.py)
```python
parts = line.split()       # Split log line by whitespace
log_level = parts[2]       # Index 2 = log level (INFO/ERROR/WARN/DEBUG)
print(f"{log_level}\t1")  # Emit (log_level, 1)
```
- Reads each line from stdin (Hadoop sends lines one by one)
- Splits by whitespace: `[date, time, LEVEL, service, -, message...]`
- `parts[2]` = the log level at index 2
- Emits `log_level TAB 1` to stdout

### Reducer (reducer.py)
```python
key, value = line.split("\t", 1)   # Split into key and value
if current_key == key:
    current_count += value          # Same key: add to count
else:
    print(f"{current_key}\t{current_count}")  # New key: emit previous
```
- Receives sorted `(log_level, 1)` pairs from Mapper
- Groups consecutive same keys and sums their values
- Emits `(log_level, total_count)` for each group

---

## WHY HADOOP STREAMING?
- Hadoop Streaming allows writing Mapper/Reducer in **any language** (Python, Ruby, etc.)
- It uses **stdin/stdout** to communicate between Hadoop and the scripts
- Hadoop sends input lines to Mapper via stdin
- Mapper writes output to stdout → Hadoop collects it
- After shuffle & sort, Reducer receives grouped data via stdin

---

## KEY HDFS COMMANDS USED
```bash
hadoop fs -mkdir -p /loginput          # Create input directory in HDFS
hadoop fs -put system_log.txt /loginput # Upload log file to HDFS
hadoop fs -ls /loginput                # List files in HDFS directory
hadoop fs -cat /logoutput/part-00000   # View MapReduce output
hadoop fs -rm -r /logoutput            # Delete output dir (before re-run)
```

---

## DIFFERENCE FROM SALES COUNTRY PRACTICAL

| Feature | Sales Country | Log Processing |
|---|---|---|
| Input | CSV file | Log file (space-separated) |
| Language | Java | Python (Hadoop Streaming) |
| Key extracted | Column 7 (country) | Column 2 (log level) |
| Split method | `split(",")` | `split()` (whitespace) |
| Output | Sales count per country | Log count per level |

---

## POSSIBLE ORAL EXAM QUESTIONS AND ANSWERS

**Q: What is this practical about?**  
A: It counts the number of occurrences of each log level (INFO, ERROR, WARN, DEBUG) from a system log file using Hadoop MapReduce distributed processing.

**Q: What does the Mapper do?**  
A: It reads each log line, splits it by whitespace, extracts the log level from index 2, and emits `(log_level, 1)` as a key-value pair.

**Q: What does the Reducer do?**  
A: It receives all 1s grouped by log level (after shuffle & sort), sums them up, and emits `(log_level, total_count)` as the final output.

**Q: What is Hadoop Streaming?**  
A: Hadoop Streaming is a utility that allows writing Mapper and Reducer in any language (like Python). It uses stdin/stdout to pass data between Hadoop and the scripts.

**Q: What is the Shuffle and Sort phase?**  
A: After the Map phase, Hadoop automatically groups all values with the same key together and sorts them. This is called Shuffle & Sort. The Reducer then processes each group.

**Q: Why do we use `parts[2]` to extract the log level?**  
A: The log format is `DATE TIME LEVEL SERVICE - MESSAGE`. After splitting by whitespace, index 0=date, 1=time, 2=log level. So `parts[2]` gives us INFO/ERROR/WARN/DEBUG.

**Q: What is HDFS?**  
A: HDFS (Hadoop Distributed File System) is Hadoop's storage system. It splits large files into blocks and distributes them across multiple nodes for parallel processing.

**Q: Why do we delete the output directory before re-running?**  
A: Hadoop does not overwrite existing output directories. If `/logoutput` already exists, the job fails. We must delete it with `hadoop fs -rm -r /logoutput` before re-running.

**Q: What is part-00000?**  
A: It is the default output file name created by the Reducer. If there are multiple Reducers, there will be part-00000, part-00001, etc.

**Q: What is a distributed application?**  
A: A distributed application runs across multiple computers (nodes) simultaneously, sharing the workload. Hadoop MapReduce distributes data processing across many nodes in a cluster for faster execution.
